In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import sys
import os
from pathlib import Path

In [ ]:
# Add src to path for imports
os.chdir(os.path.dirname(os.path.abspath('.'))) 

In [ ]:
from src.battery.battery import BatteryConfig
from src.battery.lp_optimizer import LPOptimizer
from src.battery.fetcher import compute_price_statistics

── 0. Configuration ─────────────────────────────────────────────────────────

In [ ]:
battery = BatteryConfig(
    power_mw=100.0,
    capacity_mwh=400.0,
    rte_charge=0.93,
    rte_discharge=0.95,
    degradation_cost=10.0,
    soc_min_pct=0.05,
    soc_max_pct=0.95,
    soc_initial_pct=0.50,
)

In [ ]:
optimizer = LPOptimizer(battery, dt_hours=1.0)

In [ ]:
print("Battery configuration:")
print(f"  {battery}")
print(f"  Minimum profitable spread: ${battery.minimum_profitable_spread():.2f}/MWh")
print(f"  Usable capacity: {battery.usable_capacity_mwh:.0f} MWh")

── 1. Load Prices ────────────────────────────────────────────────────────────

In [ ]:
from src.battery.fetcher import fetch_caiso_prices, fetch_ercot_prices, process_ercot_data
# prices_ercot = fetch_ercot_prices("2025-01-01", "2025-12-31")
# ERCOT API does not provide historical data that far back

prices_caiso = fetch_caiso_prices("2025-01-01", "2025-12-31")

In [ ]:
data_raw = Path("__file__").resolve().parent / "data" / "raw"
df_ercot = process_ercot_data(data_raw)
prices_ercot = df_ercot["LMP"]

In [ ]:
print(f"\nCAISO prices:")
print(f"  Min: ${prices_caiso.min():.1f}/MWh at hour {prices_caiso.idxmin().hour}")
print(f"  Max: ${prices_caiso.max():.1f}/MWh at hour {prices_caiso.idxmax().hour}")
print(f"  Spread: ${prices_caiso.max() - prices_caiso.min():.1f}/MWh")

In [ ]:
print(f"\nERCOT prices:")
print(f"  Min: ${prices_ercot.min():.1f}/MWh at hour {prices_ercot.idxmin().hour}")
print(f"  Max: ${prices_ercot.max():.1f}/MWh at hour {prices_ercot.idxmax().hour}")
print(f"  Spread: ${prices_ercot.max() - prices_ercot.min():.1f}/MWh")

In [ ]:
# NaN analysis
print(prices_caiso.resample('D').apply(lambda x: x.isna().sum()).value_counts())
prices_caiso = prices_caiso.ffill()

── 2. Solve the LP ───────────────────────────────────────────────────────────

In [ ]:
print("\nSolving LP for CAISO year...")

In [ ]:
result = optimizer.solve(prices_caiso.values)
print(f"Status: {result.status}")
print(f"Total revenue: ${result.total_revenue:.2f}")
print(f"Cycles: {result.n_cycles:.2f}")

In [ ]:
print("\nSolving LP for ERCOT year...")

In [ ]:
result = optimizer.solve(prices_ercot.values)
print(f"Status: {result.status}")
print(f"Total revenue: ${result.total_revenue:.2f}")
print(f"Cycles: {result.n_cycles:.2f}")

── 3. Plot: SOC + Price Overlay ─────────────────────────────────────────────

In [ ]:
def plot_dispatch(prices: pd.Series, result, battery: BatteryConfig, title: str):
    """
    Standard dispatch visualization: prices on top, SOC on bottom.

    This is the primary diagnostic plot. For any result you generate,
    you should be able to look at this plot and understand why the battery
    made the decisions it did.
    """
    fig = plt.figure(figsize=(14, 8))
    gs = gridspec.GridSpec(3, 1, height_ratios=[2, 1.5, 1])

    hours = np.arange(len(prices))

    # Panel 1: Prices
    ax1 = fig.add_subplot(gs[0])
    ax1.plot(hours, prices.values, 'k-', linewidth=2, label='Price')
    ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax1.fill_between(hours, prices.values, 0,
                     where=(prices.values < 0), alpha=0.3, color='red',
                     label='Negative price')
    ax1.set_ylabel('Price ($/MWh)')
    ax1.set_title(title)
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    if result is not None:
        # Panel 2: Charge/Discharge
        ax2 = fig.add_subplot(gs[1], sharex=ax1)
        ax2.bar(hours, result.discharge_mw, color='green', alpha=0.7,
                label='Discharge (revenue)')
        ax2.bar(hours, -result.charge_mw, color='red', alpha=0.7,
                label='Charge (cost)')
        ax2.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
        ax2.set_ylabel('Power (MW)')
        ax2.legend()
        ax2.grid(True, alpha=0.3)

        # Panel 3: SOC
        ax3 = fig.add_subplot(gs[2], sharex=ax1)
        ax3.fill_between(range(len(result.soc_mwh)), result.soc_mwh,
                         battery.soc_min_mwh, alpha=0.5, color='blue')
        ax3.axhline(y=battery.soc_max_mwh, color='red', linestyle='--',
                    alpha=0.5, label=f'SOC max ({battery.soc_max_mwh:.0f} MWh)')
        ax3.axhline(y=battery.soc_min_mwh, color='orange', linestyle='--',
                    alpha=0.5, label=f'SOC min ({battery.soc_min_mwh:.0f} MWh)')
        ax3.set_ylabel('SOC (MWh)')
        ax3.set_xlabel('Hour')
        ax3.legend()
        ax3.grid(True, alpha=0.3)

    plt.tight_layout()
    return fig

In [ ]:
# Plot synthetic prices even before LP is implemented
fig = plot_dispatch(prices_caiso, result=None, battery=battery,
                    title="CAISO Sample Year — Prices Only")
plt.savefig("results/plots/01_caiso_prices.png", dpi=150, bbox_inches='tight')
print("\nSaved price plot to results/plots/01_caiso_prices.png")

── 4. Revenue Metrics ────────────────────────────────────────────────────────

In [ ]:
print(f"\nRevenue metrics:")
print(f"  Total revenue: ${result.total_revenue:.2f}")
print(f"  Annualized ($/kW-yr): ${result.revenue_per_kw_yr:.2f}")
print(f"  Number of cycles: {result.n_cycles:.2f}")

In [ ]:
# Price statistics (doesn't require LP)
print("\nCAISO price statistics:")
stats = compute_price_statistics(prices_caiso)
for k, v in stats.items():
    print(f"  {k}: {v:.2f}")

── 5. CAISO vs ERCOT Comparison ─────────────────────────────────────────────

In [ ]:
# # TO DO
# result_ercot = optimizer.solve(prices_ercot.values)
# result_caiso = optimizer.solve(prices_caiso.values)

# print(f"\nMarket comparison (single day):")
# print(f"  {'Metric':<30} {'ERCOT':>12} {'CAISO':>12}")
# print(f"  {'-'*54}")
# print(f"  {'Revenue ($)':<30} {result_ercot.total_revenue:>12.2f} {result_caiso.total_revenue:>12.2f}")
# print(f"  {'Cycles':<30} {result_ercot.n_cycles:>12.2f} {result_caiso.n_cycles:>12.2f}")
# print(f"  {'Price spread ($/MWh)':<30} ...")
